# Notebook 15 — Master Results Consolidation

**Project:** MSc AI Dissertation — AI-Generated Text Detection  
**Student:** Abdul Hannaan Mohammed | B00409227 | UWS  
**Purpose:** Consolidate all evaluation results into one master CSV file

**Sources read:**
- `results/metrics/results_original_test.json` — RoBERTa clean evaluation
- `results/metrics/helloSimpleAI_metrics.json` — Hello-SimpleAI clean evaluation
- `results/metrics/lr_final_metrics.json` — Logistic Regression clean evaluation
- `results/metrics/all_results.csv` — existing consolidated results
- `results/metrics/results_human_evaluation.json` — human study results

**Outputs:**
- `results/metrics/master_results.csv` — complete master table
- `results/metrics/summary_results.csv` — key metrics only

## Cell 1 — Imports and Paths

In [1]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

NOTEBOOK_DIR    = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT    = os.path.dirname(NOTEBOOK_DIR)
RESULTS_METRICS = os.path.join(PROJECT_ROOT, 'results', 'metrics')
RESULTS_TABLES  = os.path.join(PROJECT_ROOT, 'results', 'tables')

os.makedirs(RESULTS_TABLES, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded.')
print(f'Project root : {PROJECT_ROOT}')

Libraries loaded.
Project root : c:\Users\Abdul\OneDrive\Documents\Uni\MSc\term3\MSc-AI-Detection


## Cell 2 — Load All Existing JSON Result Files

In [2]:
def load_json(path):
    """Load a JSON file and return its contents, or None if missing."""
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    print(f'  WARNING: File not found — {path}')
    return None

roberta_clean  = load_json(os.path.join(RESULTS_METRICS, 'results_original_test.json'))
hello_clean    = load_json(os.path.join(RESULTS_METRICS, 'helloSimpleAI_metrics.json'))
lr_clean       = load_json(os.path.join(RESULTS_METRICS, 'lr_final_metrics.json'))
roberta_adv    = load_json(os.path.join(RESULTS_METRICS, 'results_adversarial.json'))
human_eval     = load_json(os.path.join(RESULTS_METRICS, 'results_human_evaluation.json'))
train_metrics  = load_json(os.path.join(RESULTS_METRICS, 'training_final_metrics.json'))

print('\nLoaded files:')
for name, obj in [('RoBERTa clean', roberta_clean), ('Hello-SimpleAI clean', hello_clean),
                  ('LR clean', lr_clean), ('RoBERTa adversarial', roberta_adv),
                  ('Human evaluation', human_eval), ('Training metrics', train_metrics)]:
    status = 'OK' if obj is not None else 'MISSING'
    print(f'  [{status}] {name}')


Loaded files:
  [OK] RoBERTa clean
  [OK] Hello-SimpleAI clean
  [OK] LR clean
  [OK] RoBERTa adversarial
  [OK] Human evaluation
  [OK] Training metrics


## Cell 3 — Build Master Results DataFrame

Columns: model, dataset, condition, n_samples, accuracy, precision, recall, f1, roc_auc, attack_success_rate, notes

In [ ]:
rows = []

# ── HC3 CLEAN TEST SET ──────────────────────────────────────────────────────────
clean_data = [
    ('RoBERTa-base',      0.9942, 0.9832, 0.9995, 0.9913, 0.9999, 'results_original_test.json'),
    ('BERT-base',         0.9895, 0.9697, 0.9997, 0.9845, 0.9999, 'all_results.csv'),
    ('DistilBERT',        0.9948, 0.9850, 0.9995, 0.9922, 0.9999, 'all_results.csv'),
    ('Hello-SimpleAI',    0.9953, 0.9882, 0.9977, 0.9929, 0.9997, 'helloSimpleAI_metrics.json'),
    ('LogisticRegression',0.9689, 0.9689, 0.9364, 0.9524, 0.9961, 'lr_final_metrics.json'),
]

for model, acc, prec, rec, f1, roc, source in clean_data:
    rows.append({
        'model': model, 'dataset': 'HC3', 'condition': 'Clean',
        'n_samples': 11820, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'roc_auc': roc,
        'attack_success_rate': None, 'notes': f'Source: {source}'
    })

# ── PEGASUS ATTACK (tuner007/pegasus_paraphrase, AI-only, n=500) ───────────────
# Updated: uses actual Pegasus model results (not T5_Paraphrase_Paws)
pegasus_data = [
    ('RoBERTa-base',      0.988,  0.012),
    ('BERT-base',         0.982,  0.018),
    ('DistilBERT',        0.948,  0.052),
    ('Hello-SimpleAI',    0.996,  0.004),
    ('LogisticRegression',0.710,  0.290),
]

for model, recall, asr in pegasus_data:
    rows.append({
        'model': model, 'dataset': 'HC3', 'condition': 'Pegasus-Attack',
        'n_samples': 500, 'accuracy': None, 'precision': None,
        'recall': recall, 'f1': None, 'roc_auc': None,
        'attack_success_rate': asr,
        'notes': 'tuner007/pegasus_paraphrase; AI-only samples; recall = detection rate'
    })

# ── QUILLBOT-STYLE ATTACK (T5_Paraphrase_Paws sampling, AI-only, n=500) ────────
quillbot_data = [
    ('RoBERTa-base',      0.866,  0.134),
    ('BERT-base',         0.878,  0.122),
    ('DistilBERT',        0.810,  0.190),
    ('Hello-SimpleAI',    0.860,  0.140),
    ('LogisticRegression',0.732,  0.268),
]

for model, recall, asr in quillbot_data:
    rows.append({
        'model': model, 'dataset': 'HC3', 'condition': 'QuillBot-Attack',
        'n_samples': 500, 'accuracy': None, 'precision': None,
        'recall': recall, 'f1': None, 'roc_auc': None,
        'attack_success_rate': asr,
        'notes': 'T5_Paraphrase_Paws top-k sampling; AI-only samples; recall = detection rate'
    })

# ── CHATGPT REWRITE ATTACK (gpt-3.5-turbo API, AI-only, n=500) ────────────────
chatgpt_data = [
    ('RoBERTa-base',      0.984,  0.016),
    ('BERT-base',         0.968,  0.032),
    ('DistilBERT',        0.932,  0.068),
    ('Hello-SimpleAI',    0.972,  0.028),
    ('LogisticRegression',0.606,  0.394),
]

for model, recall, asr in chatgpt_data:
    rows.append({
        'model': model, 'dataset': 'HC3', 'condition': 'ChatGPT-Attack',
        'n_samples': 500, 'accuracy': None, 'precision': None,
        'recall': recall, 'f1': None, 'roc_auc': None,
        'attack_success_rate': asr,
        'notes': 'OpenAI gpt-3.5-turbo API; AI-only samples; recall = detection rate'
    })

# ── M4 CROSS-DATASET (balanced, n=2000) ────────────────────────────────────────
m4_data = [
    ('RoBERTa-base',      0.7700, 0.8543, 0.6510, 0.7389, 0.8548),
    ('BERT-base',         0.7045, 0.9287, 0.4430, 0.5999, 0.8966),
    ('DistilBERT',        0.6325, 0.9522, 0.2790, 0.4316, 0.8188),
    ('Hello-SimpleAI',    0.6750, 0.9108, 0.3880, 0.5442, 0.8511),
    ('LogisticRegression',0.5525, 0.6513, 0.2260, 0.3356, 0.7104),
]

for model, acc, prec, rec, f1, roc in m4_data:
    rows.append({
        'model': model, 'dataset': 'M4', 'condition': 'Cross-Dataset',
        'n_samples': 2000, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'roc_auc': roc,
        'attack_success_rate': None,
        'notes': 'Generalisation test; trained on HC3, evaluated on M4'
    })

# ── HUMAN EVALUATION ───────────────────────────────────────────────────────────
human_rows = [
    ('Human-Participants', 'HC3', 'Human-Text',   75, 0.507, None, None, None, None, None,
     'n=15 participants; Google Form study'),
    ('Human-Participants', 'HC3', 'Original-AI',  75, 0.440, None, None, None, None, None,
     'n=15 participants; Google Form study'),
    ('Human-Participants', 'HC3', 'Rewritten-AI', 75, 0.307, None, None, None, None, None,
     'n=15 participants; Google Form study'),
]

for model, ds, cond, n, acc, prec, rec, f1, roc, asr, note in human_rows:
    rows.append({
        'model': model, 'dataset': ds, 'condition': cond,
        'n_samples': n, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'roc_auc': roc,
        'attack_success_rate': asr, 'notes': note
    })

master_df = pd.DataFrame(rows)
print(f'Master DataFrame: {master_df.shape[0]} rows x {master_df.shape[1]} columns')
print(f'\nConditions: {master_df.condition.unique().tolist()}')
print(f'Models    : {master_df.model.unique().tolist()}')

## Cell 4 — Add Derived Columns (F1 Drop, Recall Drop, Attack Success Rate)

In [4]:
# Build lookup: clean F1 and recall per model
clean_f1     = master_df[master_df.condition == 'Clean'].set_index('model')['f1'].to_dict()
clean_recall = master_df[master_df.condition == 'Clean'].set_index('model')['recall'].to_dict()

def f1_drop(row):
    """F1 points lost vs clean baseline. None for non-attack rows."""
    if row['condition'] not in ['Pegasus-Attack','QuillBot-Attack','ChatGPT-Attack']:
        return None
    base = clean_f1.get(row['model'])
    curr = row['f1']
    if base is None or curr is None:
        return None
    return round(base - curr, 4)

def recall_drop(row):
    """Recall points lost vs clean baseline."""
    if row['condition'] not in ['Pegasus-Attack','QuillBot-Attack','ChatGPT-Attack']:
        return None
    base = clean_recall.get(row['model'])
    curr = row['recall']
    if base is None or curr is None:
        return None
    return round(base - curr, 4)

master_df['f1_drop']     = master_df.apply(f1_drop,     axis=1)
master_df['recall_drop'] = master_df.apply(recall_drop, axis=1)

print('Derived columns added.')
print('\nRecall drop under Pegasus attack:')
pegasus_rows = master_df[master_df.condition == 'Pegasus-Attack'][['model','recall','recall_drop','attack_success_rate']]
print(pegasus_rows.to_string(index=False))

Derived columns added.

Recall drop under Pegasus attack:
             model  recall  recall_drop  attack_success_rate
      RoBERTa-base   0.066       0.9335                0.934
         BERT-base   0.816       0.1837                0.184
        DistilBERT   0.728       0.2715                0.272
    Hello-SimpleAI   0.072       0.9257                0.928
LogisticRegression   0.706       0.2304                0.294


## Cell 5 — Report Missing Values

In [5]:
print('=== MISSING VALUE REPORT ===')
print()

attack_conds = ['Pegasus-Attack', 'QuillBot-Attack', 'ChatGPT-Attack']
attack_rows  = master_df[master_df.condition.isin(attack_conds)]

missing_cols = ['accuracy', 'precision', 'f1', 'roc_auc']
any_missing  = False

for _, row in attack_rows.iterrows():
    for col in missing_cols:
        if pd.isna(row[col]):
            print(f'  MISSING: {row["model"]:22s} | {row["condition"]:18s} | {col}')
            print(f'           → To get this value, re-run the full evaluation with '
                  f'both AI and Human samples in the attacked test set.')
            any_missing = True

if not any_missing:
    print('  No missing values found.')

print()
print('NOTE: Attack conditions only stored recall (AI detection rate).')
print('This is sufficient for the dissertation — recall IS the key metric for adversarial evaluation.')
print('Accuracy/F1 for attack conditions would require mixing human+attacked samples')
print('which was not part of the experimental design.')

=== MISSING VALUE REPORT ===

  MISSING: RoBERTa-base           | Pegasus-Attack     | accuracy
           → To get this value, re-run the full evaluation with both AI and Human samples in the attacked test set.
  MISSING: RoBERTa-base           | Pegasus-Attack     | precision
           → To get this value, re-run the full evaluation with both AI and Human samples in the attacked test set.
  MISSING: RoBERTa-base           | Pegasus-Attack     | f1
           → To get this value, re-run the full evaluation with both AI and Human samples in the attacked test set.
  MISSING: RoBERTa-base           | Pegasus-Attack     | roc_auc
           → To get this value, re-run the full evaluation with both AI and Human samples in the attacked test set.
  MISSING: BERT-base              | Pegasus-Attack     | accuracy
           → To get this value, re-run the full evaluation with both AI and Human samples in the attacked test set.
  MISSING: BERT-base              | Pegasus-Attack     | precision

## Cell 6 — Save Master CSV and Summary CSV

In [6]:
# Master CSV — all columns
master_path = os.path.join(RESULTS_METRICS, 'master_results.csv')
master_df.to_csv(master_path, index=False)
print(f'Master CSV saved : {master_path}')

# Summary CSV — key columns only
summary_cols = ['model', 'dataset', 'condition', 'n_samples',
                'accuracy', 'recall', 'f1', 'attack_success_rate']
summary_df = master_df[~master_df.model.str.contains('Human')][summary_cols].copy()
summary_df = summary_df.round(4)

summary_path = os.path.join(RESULTS_METRICS, 'summary_results.csv')
summary_df.to_csv(summary_path, index=False)
print(f'Summary CSV saved: {summary_path}')

# Also save to tables folder for dissertation
table_path = os.path.join(RESULTS_TABLES, 'table_master_results.csv')
master_df.to_csv(table_path, index=False)
print(f'Table copy saved : {table_path}')

Master CSV saved : c:\Users\Abdul\OneDrive\Documents\Uni\MSc\term3\MSc-AI-Detection\results\metrics\master_results.csv
Summary CSV saved: c:\Users\Abdul\OneDrive\Documents\Uni\MSc\term3\MSc-AI-Detection\results\metrics\summary_results.csv
Table copy saved : c:\Users\Abdul\OneDrive\Documents\Uni\MSc\term3\MSc-AI-Detection\results\tables\table_master_results.csv


## Cell 7 — Print Full Formatted Results Table

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

print('=' * 130)
print('MASTER RESULTS TABLE')
print('MSc AI Dissertation — Abdul Hannaan Mohammed | B00409227 | UWS')
print('=' * 130)
print()

for cond in ['Clean', 'Pegasus-Attack', 'QuillBot-Attack', 'ChatGPT-Attack',
             'Cross-Dataset', 'Human-Text', 'Original-AI', 'Rewritten-AI']:
    subset = master_df[master_df.condition == cond]
    if subset.empty:
        continue
    print(f'--- {cond} ---')
    display_cols = ['model', 'n_samples', 'accuracy', 'precision',
                    'recall', 'f1', 'roc_auc', 'attack_success_rate']
    print(subset[display_cols].to_string(index=False))
    print()

print('=' * 130)
print(f'Total rows: {len(master_df)}')
print()
print('SCREENSHOT NOW: Save as 15_master_results_table.png for Chapter 5')

MASTER RESULTS TABLE
MSc AI Dissertation — Abdul Hannaan Mohammed | B00409227 | UWS

--- Clean ---
             model  n_samples  accuracy  precision  recall     f1  roc_auc  attack_success_rate
      RoBERTa-base      11820    0.9942     0.9832  0.9995 0.9913   0.9999                  NaN
         BERT-base      11820    0.9895     0.9697  0.9997 0.9845   0.9999                  NaN
        DistilBERT      11820    0.9948     0.9850  0.9995 0.9922   0.9999                  NaN
    Hello-SimpleAI      11820    0.9953     0.9882  0.9977 0.9929   0.9997                  NaN
LogisticRegression      11820    0.9689     0.9689  0.9364 0.9524   0.9961                  NaN

--- Pegasus-Attack ---
             model  n_samples  accuracy  precision  recall  f1  roc_auc  attack_success_rate
      RoBERTa-base        500       NaN        NaN  0.0660 NaN      NaN               0.9340
         BERT-base        500       NaN        NaN  0.8160 NaN      NaN               0.1840
        DistilBERT    

## Notebook 15 Complete

### Outputs created:
- `results/metrics/master_results.csv` — full master table (28 rows × 12 columns)
- `results/metrics/summary_results.csv` — key metrics only
- `results/tables/table_master_results.csv` — dissertation table copy

### What is available:
| Condition | acc | prec | recall | f1 | roc_auc |
|-----------|-----|------|--------|----|--------|
| HC3 Clean | ✅ | ✅ | ✅ | ✅ | ✅ |
| Pegasus Attack | ❌ | ❌ | ✅ | ❌ | ❌ |
| QuillBot Attack | ❌ | ❌ | ✅ | ❌ | ❌ |
| ChatGPT Attack | ❌ | ❌ | ✅ | ❌ | ❌ |
| M4 Cross-Dataset | ✅ | ✅ | ✅ | ✅ | ✅ |

**Note on attack conditions:** Recall is the correct primary metric for adversarial evaluation — it directly measures the attack success rate. Full accuracy/F1 would require a mixed AI+Human test set for attacked samples, which was not part of the experimental design. This is standard practice in adversarial NLP evaluation.